In [ ]:
# Source Code 6
# Script to make predictions using the pre-trained model.

!pip install tensorflow keras opencv-python  # Installs TensorFlow, Keras (for deep learning), and OpenCV (for image processing)

import os  # Provides functions to interact with the operating system (e.g., file paths)
import cv2  # Imports OpenCV library for image and video processing
import numpy as np  # Imports NumPy for numerical operations and array handling
import pandas as pd  # Imports pandas for data manipulation and analysis
from keras.models import Model  # Imports the base Model class from Keras to build custom models
from keras.models import load_model  # Imports function to load pre-trained Keras models from disk
import shutil  # Imports shutil for high-level file operations like copying and deleting files

# Specify the path to the saved model
model_path = 'E:/mobilenet_mhc.h5'

# Load the model
model = load_model(model_path)

# Initialize empty lists to store results
file_names = []
interpretations = []
probabilities = []

# Folder containing the aerial (SATELLTE) images
folder_path = 'E:/wi/sat'

# Output folder for copied images
output_folder = 'E:/wi/sat-detected'

# Output folder for DataFrame as a CSV file
csv_path = 'E:/wi/mhc_detected.csv'

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Loop through each image in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.tif'):
        image_path = os.path.join(folder_path, file_name)
        print("Processing image:", image_path)
        
        # Load and preprocess the image
        img = cv2.imread(image_path)
        img = cv2.resize(img, (224, 224))
        img = img.astype('float32') / 255
        img = np.expand_dims(img, axis=0)
        
        # Make predictions
        predictions = model.predict(img)
        print("Predictions:", predictions)
        
        # Interpret the predictions
        if predictions[0] >= 0.5:
            interpretation = "YES"
        else:
            interpretation = "NO"
        print("Interpretation:", interpretation)
        
        # Store results
        file_names.append(file_name)
        interpretations.append(interpretation)
        probabilities.append(predictions[0])
        
        # Copy the image to the output folder if interpretation is "YES"
        if interpretation == "YES":
            shutil.copy(image_path, os.path.join(output_folder, file_name))

# Create a DataFrame from the results
result_df = pd.DataFrame({
    'File Name': file_names,
    'Interpretation': interpretations,
    'Probability': probabilities
})

# Save the DataFrame as a CSV file
result_df.to_csv(csv_path, index=False)

print("Results saved to", csv_path)